## Calculate the PSF pack

This notebook calculates the *PSF pack* which can be used as a matched filter for calculating the position of stars in the field of view.

The main element of the PSF pack, the element `psf`, provides the PSF impulse response of the system for rays coming from differemnt directions. It is a 4-dimensional Numpy array indexed by the x,y direction of the rays and the i,j position of each pixel in the camera. The direction of the rays is defined by the position of the principal ray on the camera plane. The shape of this object is 257x257x32x32, with the direction of the rays being sampled with 1/8 pixel spacing, between -16.0 and +16.0, centered on 0.0.

The following eElements are in the PSF pack:

- `timestamp`: the date & time at which the pack was calculated
- `model_data_pack`: a copy of the optical model data pack
- `xgrid` & `ygrid`: 1D arrays describing direction principal rays in terms of impact point on image plane (shape=257)
- `xprincipal` & `yprincipal`: 2D arrays describing direction principal rays in terms of impact point on image plane (shape=257,257). These are the computed from `xgrid` and `ygrid` using the Numpy `meshgrid` function, such that $\mathrm{xprincpal}(y,x)=\mathrm{xgrid}(x)$ for $0\le x,y\le256$.
- `xmedian` & `ymedian`: the median position of the rays on the image plane (shape=257,257) for each simulated direction. Due to aberations in the lens the median position on the image plane differs from the nominal position of the principal ray.
- `rmedian`: the median offset of the rays from the center of the image plane (shape=257,257). This can be ueful for calculating th eeffective plate scale of the instrument. In general $\mathrm{rmedian}^2 \ne \mathrm{xmedian}^2 + \mathrm{ymedian}^2$.
- `xmean` & `ymean`: the mean position of the rays on the image plane (shape=257,257) for each simulated direction. Due to aberations in the lens the mean position on the image plane differs signifcantly from the nominal position of the principal ray.
- `rmean`: the mean offset of the rays from the center of the image plane (shape=257,257).
- `xQ` & `yQ`: the center of the smallest circle containing Q=50% of the rays on the image plane (shape=257,257).
- `dQ`: the diameter of the smallest circle containing Q=50% of the rays on the image plane (shape=257,257).
- `psf`: the impulse response for the system as a function of the direction if the rays (shape=257,257,32,32). If the PSF is fully contained in the image then $\sum_{i,j}\mathrm{psf}(x,y,i,j)=1$ $\forall x,y$. Otherwise the sum is less than unity.
- `psf_sum`: sum of the PSF contained in the image for each direction (shape=257,257).
- `psf_sumsq`: sum of the squared PSF contained in the image for each direction (shape=257,257`. An intriguing quantity whose significance I am yet to understand.

The notebook uses ProcessPool concurrency to speed up the calculation and benefits from being run on a computer with many cores available. The number of cores used can be restricted in the call to  the constructor of `ProcessPoolExecutor` if desired.

In [1]:
from concurrent import futures
import numpy as np
import datetime

In [ ]:
# Define function to initialize global variables as this will need to be done
# in each of the ProcessPool executors
def init():
    import sys
    sys.path.append("../..")
    from pypanodecoder import optical_model
    import numpy as np
    np.random.seed()  # reseeds the legacy global state from OS entropy
    global data_pack
    data_pack = optical_model.load_datapack()
    global xstar
    xstar = np.linspace(-16.0,16.0,32*8+1)
    global ystar
    ystar = np.linspace(-16.0,16.0,32*8+1)    

In [ ]:
# Initialize the variables here in this notbool for use below
init()

In [ ]:
# Function to compute the PSF image and stats for one single direction
def run(i):
    from pypanodecoder import optical_model
    import numpy as np
    iy,ix = np.unravel_index(i, shape=(len(ystar),len(xstar)))
    y = ystar[iy]
    x = xstar[ix]
    rnom = (x,y)
    return i,(iy,ix),rnom,*optical_model.generate_psf_image(*rnom,10000000,data_pack,calc_diameter=True, diameter_quantile=0.5)

In [ ]:
# Define variables to hold results
xnom = np.zeros((len(ystar), len(xstar)))
ynom = np.zeros((len(ystar), len(xstar)))
xmedian = np.zeros((len(ystar), len(xstar)))
ymedian = np.zeros((len(ystar), len(xstar)))
rmedian = np.zeros((len(ystar), len(xstar)))
xmean = np.zeros((len(ystar), len(xstar)))
ymean = np.zeros((len(ystar), len(xstar)))
rmean = np.zeros((len(ystar), len(xstar)))
xQ = np.zeros((len(ystar), len(xstar)))
yQ = np.zeros((len(ystar), len(xstar)))
dQ = np.zeros((len(ystar), len(xstar)))
psfpack = np.zeros((len(ystar), len(xstar), 32, 32))

In [ ]:
# Queue the tasks in the ProcessPoolExecutor and extract results to the
# appropriate variables
with futures.ProcessPoolExecutor(initializer=init) as runner:
    for result in runner.map(run, range(len(ystar)*len(xstar))):
        i, ri, rnom, nray, image, center_mean, r_mean, center_med, r_med, rQ, _dQ = result
        xnom[*ri], ynom[*ri] = rnom
        xmedian[*ri], ymedian[*ri] = center_med
        rmedian[*ri] = r_med
        xmean[*ri], ymean[*ri] = center_mean
        rmean[*ri] = r_mean
        xQ[*ri], yQ[*ri] = rQ
        dQ[*ri] = _dQ
        psfpack[*ri, ...] = image/nray

In [ ]:
# Make PSF pack dictionary
results = dict(
    timestamp=datetime.datetime.now(datetime.UTC).isoformat(),
    model_data_pack=data_pack,
    xgrid=xstar,
    ygrid=ystar,
    xprincipal=xnom,
    yprincipal=ynom,
    xmedian=xmedian,
    ymedian=ymedian,
    rmedian=rmedian,
    xmean=xmean,
    ymean=ymean,
    rmean=rmean,
    xQ=xQ, 
    yQ=yQ, 
    dQ=dQ,
    psf=psfpack,
    psf_sum=psfpack.sum(axis=(-2, -1)),
    psf_sumsq=(psfpack**2).sum(axis=(-2, -1))
)

In [ ]:
# Save PSF pack
np.savez_compressed('psf_model_pack.npz', **results)